# Data Preprocessing — Loan Approval Prediction

Turns the raw CSV into a model-ready train/test split. Cleaning and feature engineering logic lives in `src/preprocessing.py` so it can be reused identically by the modeling notebook and the Streamlit app — this notebook just calls into it and inspects the result at each step.

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone -q https://github.com/AnaNicuesa/loan-approval-prediction.git
    %cd loan-approval-prediction
    %pip install -q -r requirements.txt

In [2]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    path = start.resolve()
    for _ in range(4):
        if (path / "src").exists() and (path / "data").exists():
            return path
        path = path.parent
    raise FileNotFoundError("Could not locate project root from " + str(start))


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.preprocessing import load_raw_data, clean_data, engineer_features, build_preprocessor
from src.utils import (
    RAW_DATA_PATH,
    PROCESSED_DATA_DIR,
    TARGET,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    RANDOM_STATE,
    ensure_directories,
)

ensure_directories()
pd.set_option("display.max_columns", None)

## 1. Cleaning

`clean_data` drops exact duplicates, removes rows without a target label, drops the `loan_id` identifier, and normalizes the `dependents` column (`"3+"` becomes `3`) so it can be treated as numeric.

In [4]:
raw_df = load_raw_data(RAW_DATA_PATH)
print(f"Raw shape: {raw_df.shape}")

clean_df = clean_data(raw_df)
print(f"Clean shape: {clean_df.shape}")
clean_df.head()

Raw shape: (563, 13)
Clean shape: (510, 12)


,gender,married,dependents,education,self_employed,applicant_income,coapplicant_income,loan_amount,loan_amount_term,credit_history,property_area,loan_status
0,Male,Yes,1.0,Graduate,No,4583.0,1508.0,128.0,360.0,1.0,Rural,0
1,Male,Yes,0.0,Graduate,NaN,3000.0,0.0,66.0,360.0,1.0,Urban,1
2,Male,Yes,0.0,Not Graduate,No,2583.0,2358.0,120.0,360.0,1.0,Urban,1
3,Male,No,0.0,Graduate,No,6000.0,0.0,141.0,360.0,1.0,Urban,1
4,Male,Yes,2.0,Graduate,Yes,5417.0,4196.0,267.0,360.0,1.0,Urban,1


In [5]:
print("Remaining missing values:")
clean_df.isna().sum()

Remaining missing values:


gender                23
married               19
dependents            29
education             20
self_employed         32
applicant_income      24
coapplicant_income    31
loan_amount           30
loan_amount_term      25
credit_history        20
property_area         19
loan_status            0
dtype: int64

## 2. Feature engineering

Two derived features address what the EDA notebook flagged as a gap: raw income and loan amount barely correlate with approval on their own, so `total_income` (applicant + coapplicant) and `loan_income_ratio` (loan size relative to that income) are added to give the models an affordability signal to work with.

In [6]:
engineered_df = engineer_features(clean_df)
engineered_df[["applicant_income", "coapplicant_income", "total_income", "loan_amount", "loan_income_ratio"]].head()

,applicant_income,coapplicant_income,total_income,loan_amount,loan_income_ratio
0,4583.0,1508.0,6091.0,128.0,0.021015
1,3000.0,0.0,3000.0,66.0,0.022000
2,2583.0,2358.0,4941.0,120.0,0.024287
3,6000.0,0.0,6000.0,141.0,0.023500
4,5417.0,4196.0,9613.0,267.0,0.027775


## 3. Missing value handling and encoding — how the pipeline treats each column

Imputation, scaling, and encoding are not applied here directly. They live inside a `ColumnTransformer` (`build_preprocessor`) that gets fit exclusively on the training fold within each cross-validation split in the modeling notebook — fitting a scaler or encoder on the full dataset before splitting would leak information from the test set into training.

The cell below fits the transformer on a copy of the training data purely to show what comes out the other end: median imputation and standardization for numeric columns, most-frequent imputation and one-hot encoding for categorical columns.

In [7]:
preview_train, preview_test = train_test_split(
    engineered_df, test_size=0.2, stratify=engineered_df[TARGET], random_state=RANDOM_STATE
)

preprocessor = build_preprocessor()
transformed_preview = preprocessor.fit_transform(preview_train[NUMERIC_FEATURES + CATEGORICAL_FEATURES])
feature_names = preprocessor.get_feature_names_out()

print(f"Transformed shape: {transformed_preview.shape}")
pd.DataFrame(transformed_preview[:5], columns=feature_names)

Transformed shape: (408, 19)


,numeric__applicant_income,numeric__coapplicant_income,numeric__loan_amount,numeric__loan_amount_term,numeric__credit_history,numeric__dependents,numeric__total_income,numeric__loan_income_ratio,categorical__gender_Female,categorical__gender_Male,categorical__married_No,categorical__married_Yes,categorical__education_Graduate,categorical__education_Not Graduate,categorical__self_employed_No,categorical__self_employed_Yes,categorical__property_area_Rural,categorical__property_area_Semiurban,categorical__property_area_Urban
0,0.560853,-0.609799,1.026155,0.278486,0.352089,-0.755684,0.323755,-0.109439,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1,1.676489,-0.609799,0.580365,0.278486,-2.840188,-0.755684,1.374808,-0.110645,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
2,-0.251085,0.580604,0.159341,0.278486,0.352089,0.238894,-0.531665,-0.107304,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
3,-0.532368,0.199309,-0.137853,0.278486,0.352089,-0.755684,-0.382227,-0.108868,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0.119860,-0.151952,-0.212151,0.278486,0.352089,-0.755684,-0.091709,-0.109858,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


## 4. Train / test split

An 80/20 stratified split keeps the ~71/29 approval imbalance consistent between the two sets. `random_state` is fixed so the split — and everything downstream of it — is reproducible.

In [8]:
feature_columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = engineered_df[feature_columns]
y = engineered_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Train target balance:")
print(y_train.value_counts(normalize=True))
print("Test target balance:")
print(y_test.value_counts(normalize=True))

Train: (408, 13), Test: (102, 13)
Train target balance:
loan_status
1    0.713235
0    0.286765
Name: proportion, dtype: float64
Test target balance:
loan_status
1    0.715686
0    0.284314
Name: proportion, dtype: float64


## 5. Saving the processed split

The saved CSVs hold cleaned, feature-engineered, but still human-readable values (no scaling or one-hot columns baked in). Encoding and scaling are re-applied inside the modeling pipeline so they're always fit fresh on whatever training fold is active — this is what gets loaded by `03_Modeling.ipynb`.

In [9]:
train_out = X_train.copy()
train_out[TARGET] = y_train
test_out = X_test.copy()
test_out[TARGET] = y_test

train_out.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
test_out.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)

print(f"Saved train.csv ({train_out.shape}) and test.csv ({test_out.shape}) to {PROCESSED_DATA_DIR.relative_to(PROJECT_ROOT)}")

Saved train.csv ((408, 14)) and test.csv ((102, 14)) to data/processed


## 6. Summary

- 563 raw rows reduce to a smaller, deduplicated set once exact duplicates and unlabeled rows are dropped.
- `dependents` is cast to numeric and two affordability features are engineered from income and loan amount.
- Imputation, scaling, and one-hot encoding are deferred to a `ColumnTransformer` fit inside the modeling pipeline, not applied globally here, to keep the train/test boundary leakage-free.
- The stratified 80/20 split and both processed CSVs are ready for `03_Modeling.ipynb`.

> **Potential Interview Questions**
>
> **Q: Why fit the `ColumnTransformer` inside the modeling pipeline instead of transforming the whole dataset once, here, before the split?**
> A: Fitting the imputer, scaler, and encoder on the full dataset before splitting would let statistics from the test set (medians, category frequencies) leak into how the training data gets transformed. Fitting it only inside the pipeline, on each training fold, keeps the test set genuinely unseen until the final evaluation.
>
> **Q: Why median imputation for numeric features and most-frequent for categorical ones, instead of something more sophisticated like KNN or iterative imputation?**
> A: With missingness scattered at 3-6% per column and no evidence from the EDA notebook that it's anything other than close to random, a simple, fast, well-understood imputer is a reasonable default. A more complex imputation strategy would add model complexity and a new source of leakage risk without clear evidence it's needed here.
>
> **Q: Why engineer `total_income` and `loan_income_ratio` instead of feeding `applicant_income` and `coapplicant_income` directly and letting the model figure it out?**
> A: The EDA notebook showed raw income barely correlates with approval on its own. A tree-based model could in principle learn an income-to-loan ratio from splits on the raw columns, but making it an explicit feature makes that signal available to every model immediately, including linear ones, rather than depending on each model rediscovering it independently.
>
> **Q: What happens to `dependents="3+"` at prediction time in the app, where the pipeline was never fit on that exact string?**
> A: It's converted to the numeric value `3` before it reaches the pipeline at all -- `src/predict.py`'s `build_input_frame` and this notebook's `clean_data` both apply the same `"3+"` to `3` conversion, so the pipeline only ever sees the numeric column, whether the data comes from this notebook or a live form submission.